# Module 3 (CrewAI) Exercises - Mikeaig4real

This notebook mirrors the module exercise workflow for `3_crew` by implementing practical customizations aligned with the project READMEs:

- define specialized agents,
- define explicit tasks,
- orchestrate them in a crew,
- run with configurable inputs,
- generate a final report artifact.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from typing import List

from dotenv import load_dotenv
from crewai import Agent, Crew, Process, Task


In [ ]:
load_dotenv(override=True)


## Exercise 1: Debate Crew adaptation

Build a two-sided debate flow and a moderator that returns a concise decision memo.


In [ ]:
pro_agent = Agent(
    role="Pro Analyst",
    goal="Argue in favor of adopting retrieval-augmented generation for enterprise support bots.",
    backstory="You are a pragmatic architect focused on measurable delivery outcomes.",
    verbose=False,
)

con_agent = Agent(
    role="Con Analyst",
    goal="Argue against immediate adoption and surface operational risks.",
    backstory="You are a risk-focused architect who prevents expensive platform mistakes.",
    verbose=False,
)

moderator_agent = Agent(
    role="Debate Moderator",
    goal="Synthesize both sides into an implementation recommendation.",
    backstory="You create balanced recommendation memos for engineering leaders.",
    verbose=False,
)

pro_task = Task(
    description="Create 5 concise points supporting RAG adoption for enterprise support bots.",
    expected_output="A numbered list of 5 pro-adoption arguments.",
    agent=pro_agent,
)

con_task = Task(
    description="Create 5 concise points warning against rushed RAG adoption.",
    expected_output="A numbered list of 5 risk arguments.",
    agent=con_agent,
)

moderator_task = Task(
    description=(
        "Review the pro and con arguments and produce a decision memo with: "
        "(1) decision, (2) phased rollout plan, (3) risk controls."
    ),
    expected_output="A decision memo under 300 words.",
    agent=moderator_agent,
    context=[pro_task, con_task],
)

debate_crew = Crew(
    agents=[pro_agent, con_agent, moderator_agent],
    tasks=[pro_task, con_task, moderator_task],
    process=Process.sequential,
    verbose=False,
)


## Exercise 2: Engineering Team adaptation

Create a compact engineering planning crew that produces a sprint-ready implementation outline.


In [ ]:
@dataclass
class FeatureBrief:
    name: str
    audience: str
    success_metric: str


brief = FeatureBrief(
    name="Agentic incident triage assistant",
    audience="SRE and on-call engineers",
    success_metric="Reduce mean time to triage by 30%",
)

architect = Agent(
    role="Software Architect",
    goal="Design a reliable architecture and identify service boundaries.",
    backstory="You specialize in backend architecture and integration patterns.",
    verbose=False,
)

qa_lead = Agent(
    role="QA Lead",
    goal="Define acceptance criteria and critical test scenarios.",
    backstory="You prevent regressions with explicit test strategy and edge-case coverage.",
    verbose=False,
)

delivery_manager = Agent(
    role="Delivery Manager",
    goal="Produce a sprint-ready implementation plan.",
    backstory="You convert technical plans into scoped execution steps.",
    verbose=False,
)

architecture_task = Task(
    description=(
        f"Design architecture for '{brief.name}' used by {brief.audience}. "
        f"Primary metric: {brief.success_metric}."
    ),
    expected_output="Architecture summary with components, APIs, and deployment notes.",
    agent=architect,
)

testing_task = Task(
    description="Write acceptance criteria and top 10 QA scenarios for launch readiness.",
    expected_output="Checklist with acceptance criteria and test scenarios.",
    agent=qa_lead,
    context=[architecture_task],
)

planning_task = Task(
    description="Create a 2-sprint delivery plan with milestones, owners, and dependencies.",
    expected_output="Sprint plan with milestone table and risk register.",
    agent=delivery_manager,
    context=[architecture_task, testing_task],
)

engineering_crew = Crew(
    agents=[architect, qa_lead, delivery_manager],
    tasks=[architecture_task, testing_task, planning_task],
    process=Process.sequential,
    verbose=False,
)


## Exercise 3: Financial Researcher adaptation

Run a constrained equity scan with transparent assumptions and a final recommendation report.


In [ ]:
market_researcher = Agent(
    role="Market Researcher",
    goal="Summarize macro and sector context for US cloud software equities.",
    backstory="You produce concise context from recent market signals.",
    verbose=False,
)

fundamental_analyst = Agent(
    role="Fundamental Analyst",
    goal="Compare valuation, growth, and profitability indicators.",
    backstory="You create balanced investment analysis with clear caveats.",
    verbose=False,
)

portfolio_reviewer = Agent(
    role="Portfolio Reviewer",
    goal="Provide a recommendation with explicit risk controls.",
    backstory="You avoid overconfident conclusions and document assumptions.",
    verbose=False,
)

market_task = Task(
    description="Summarize the current environment for large-cap US cloud software stocks.",
    expected_output="Macro + sector context in bullet points.",
    agent=market_researcher,
)

fundamental_task = Task(
    description="Compare MSFT, CRM, and NOW across growth, margins, and valuation factors.",
    expected_output="Comparison table with tradeoffs and data caveats.",
    agent=fundamental_analyst,
    context=[market_task],
)

review_task = Task(
    description=(
        "Provide one recommendation for a 6-12 month horizon with downside risks, "
        "position sizing guidance, and no guarantee language."
    ),
    expected_output="Recommendation memo with rationale and risk controls.",
    agent=portfolio_reviewer,
    context=[market_task, fundamental_task],
)

financial_crew = Crew(
    agents=[market_researcher, fundamental_analyst, portfolio_reviewer],
    tasks=[market_task, fundamental_task, review_task],
    process=Process.sequential,
    verbose=False,
)


In [ ]:
# Run one crew at a time.
# result = debate_crew.kickoff()
# result = engineering_crew.kickoff()
# result = financial_crew.kickoff()

print("Ready: debate_crew, engineering_crew, financial_crew")


In [ ]:
# Optional: write the latest result to a dated report file.

def save_report(text: str, prefix: str = "week3_crew_report") -> str:
    """Persist a crew output to markdown."""
    stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    path = f"{prefix}_{stamp}.md"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path
